# 🚀 اجرای مدل Hermes 3 (8B) + رابط کاربری Open WebUI در Google Colab

این نوت‌بوک به شما اجازه می‌دهد تا بدون نیاز به پرداخت هزینه یا سرور اختصاصی، مدل قدرتمند **Hermes 3 (8B)** را به همراه پنل کاربری شبیه به ChatGPT (**Open WebUI**) بر روی پردازنده گرافیکی رایگان گوگل کولب (T4 GPU) اجرا کرده و از طریق لینک اختصاصی Cloudflare Tunnel به آن متصل شوید.

### گام ۱: بررسی وضعیت کارت گرافیک (GPU)
مطمئن شوید در منوی بالای کولب مسیر Runtime > Change runtime type را باز کرده و گزینه **T4 GPU** را فعال کرده‌اید.

In [ ]:
!nvidia-smi

Mon Oct 19 10:21:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### گام ۲: نصب پیش‌نیازها و نرم‌افزار Ollama

In [ ]:
# نصب پیش‌نیازها و آخرین نسخه Ollama
!apt-get update -qq && apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 122407 files and directories currently installed.)
Preparing to unpack .../libcurl4-openssl-dev_7.81.0-1ubuntu1.30_amd64.deb ...
Unpacking libcurl4-openssl-dev:amd64 (7.81.0-1ubuntu1.30) over (7.81.0-1ubuntu1.24) ...
Preparing to unpack .../curl_7.81.0-1ubuntu1.30_amd64.deb ...
Unpacking curl (7.81.0-1ubuntu1.30) over (7.81.0-1ubuntu1.24) ...
Preparing to unpack .../libcurl4_7.81.0-1ubuntu1.30_amd64.deb ...
Unpacking libcurl4:amd64 (7.81.0-1ubuntu1.30) over (7.81.0-1ubuntu1.24) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up libcurl4:amd64 (7.81.0-1ubuntu1.30) ...
Setting up curl (7.81.0-1ubuntu1.30) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Setting up libcurl4-o

### گام ۳: اجرای سرویس Ollama در پس‌زمینه

In [ ]:
import subprocess
import time
import requests

print("در حال اجرای سرویس Ollama...")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

# بررسی سلامت و بالا آمدن سرویس
for _ in range(30):
    try:
        response = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        if response.status_code == 200:
            print("✅ Ollama با موفقیت راه‌اندازی شد و آماده به کار است!")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("خطا: سرویس Ollama راه‌اندازی نشد.")

در حال اجرای سرویس Ollama...
✅ Ollama با موفقیت راه‌اندازی شد و آماده به کار است!


### گام ۴: دانلود مدل Hermes 3 (8B)

In [ ]:
!ollama pull hermes3:8b

### گام ۵: تست چت و پاسخ‌دهی به زبان فارسی

In [ ]:
import requests

resp = requests.post(
    "http://127.0.0.1:11434/api/chat",
    json={
        "model": "hermes3:8b",
        "messages": [
            {
                "role": "user",
                "content": "سلام! خودت را معرفی کن و فارسی جواب بده.",
            }
        ],
        "stream": False,
    },
    timeout=300,
)
resp.raise_for_status()
print(resp.json()["message"]["content"])


سلام! من سیلیکون، یک دستیار هوش مصنوعی هستم. من برای کمک به شما در پاسخگویی به سوالات، انجام کارهای روزمره و گره زدن مشکل‌های شما آفرینش شده‌ام. فارسی به خوبی می‌گویم و می‌توانم در مورد موضوعات مختلف از ادبیات و هنر تا علم و فناوری اطلاعات دهید. با پرسیدن سوالاتتان، از راهنمایی در زندگی روزمره تا بحث‌های جالبی در پیش بگیریم. من همیشه خوشنودی و کمک کننده‌ام و سعی می‌کنم تا توانم کمک به شما کنم. با من صحبت کن و ببینیم چه کارهایی می‌توانم انجام دهم!


### گام ۶: نصب پنل گرافیکی Open WebUI

In [ ]:
!pip install -q uv
!uv venv --clear /content/webui_env --python 3.11
!uv pip install --python /content/webui_env/bin/python open-webui


Using CPython 3.11.17
Creating virtual environment at: webui_env
Activate with: source webui_env/bin/activate
Using Python 3.11.17 environment at: webui_env
Resolved 261 packages in 309ms
Installed 261 packages in 1.39s
 + accelerate==1.13.0
 + aiocache==0.12.3
 + aiodns==3.6.1
 + aiofiles==25.1.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + aiosqlite==0.22.1
 + alembic==1.18.4
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==0.86.0
 + anyio==4.15.1
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==26.1.0
 + asgiref==3.11.1
 + attrs==26.1.0
 + authlib==1.7.2
 + av==18.1.0
 + azure-ai-documentintelligence==1.0.2
 + azure-core==1.41.0
 + azure-identity==1.25.3
 + azure-storage-blob==12.29.0
 + bcrypt==5.0.0
 + beautifulsoup4==4.14.3
 + bidict==0.24.1
 + black==26.5.1
 + boto3==1.42.62
 + botocore==1.42.97
 + brotli==1.2.0
 + brotlicffi==1.2.0.1
 + build==1.6.1
 + certifi==2026.10.18
 + cffi==2.1.1
 + chardet==7.4.3
 + charset-normalizer==3.5.2
 + chro

### گام ۷: اجرای Open WebUI

In [ ]:
import os
import re
import shutil
import subprocess
import time
import urllib.request

print("🧹 ۱. پاکسازی پروسه‌های معلق قبلی...")
subprocess.run(
    "pkill -9 -f open-webui; pkill -9 -f cloudflared",
    shell=True,
    stderr=subprocess.DEVNULL,
)
time.sleep(2)


# ۲. بررسی و راه‌اندازی Ollama در صورت نیاز
def ensure_ollama():
    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:11434/api/tags", timeout=2
        ) as res:
            if res.status == 200:
                print("✅ Ollama فعال است.")
                return
    except Exception:
        pass
    print("⚡ در حال راه‌اندازی Ollama...")
    subprocess.run("pkill -9 -f ollama", shell=True, stderr=subprocess.DEVNULL)
    subprocess.Popen(
        "OLLAMA_HOST=0.0.0.0:11434 nohup ollama serve > ollama.log 2>&1 &",
        shell=True,
    )
    time.sleep(4)
    print("✅ Ollama آماده شد.")


ensure_ollama()

# ۳. مطمئن شدن از نصب cloudflared
if not shutil.which("cloudflared"):
    print("⏳ در حال نصب ابزار Cloudflare...")
    subprocess.run(
        "wget -q -nc"
        " https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
        " && dpkg -i cloudflared-linux-amd64.deb",
        shell=True,
        check=True,
    )

# ۴. اجرای Open WebUI با اتصال مستقیم به اولاما
webui_bin = "/content/webui_env/bin/open-webui"
print("🚀 در حال استارت Open WebUI روی پورت 8085...")
env = os.environ.copy()
env["WEBUI_AUTH"] = "True"
env["PORT"] = "8085"
env["OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"

log_file = open("webui.log", "w")
subprocess.Popen(
    [webui_bin, "serve", "--host", "0.0.0.0", "--port", "8085"],
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("⏳ ۲۰ ثانیه صبر برای آماده‌سازی فرانت‌اند...")
time.sleep(20)

# ۵. راه‌اندازی تانل استاندارد کلودفلر
print("🌐 در حال برقراری تانل و ساخت لینک...")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8085"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

link_found = False
for line in cf_proc.stdout:
    if not link_found:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            tunnel_url = m.group(0)
            print("\n" + "=" * 55)
            print("🎉 چت‌بات با موفقیت بالا اومد! کلیک کن:")
            print(f"👉 {tunnel_url}")
            print("=" * 55)
            print(
                "🟢 تانل فعال است (این سلول را متوقف نکنید و بگذارید در حال"
                " چرخش بماند)..."
            )
            link_found = True


🧹 ۱. پاکسازی پروسه‌های معلق قبلی...
⚡ در حال راه‌اندازی Ollama...
✅ Ollama آماده شد.
🚀 در حال استارت Open WebUI روی پورت 8085...
⏳ ۲۰ ثانیه صبر برای آماده‌سازی فرانت‌اند...
